---

# Dudley Economic Activity — Data Cleaning

---

## Objectives
- Inspect the dataset structure and geographical coverage.
- Check for missing values, duplicates and incorrect data types.
- Prepare the economic activity data for analysis.
- Save a cleaned copy while preserving the raw dataset.

## Inputs
- `raw_datasets/dudley_economic_activity.csv`

## Outputs

- Cleaned dataset: `cleaned_datasets/dudley_economic_activity_cleaned.csv`
- Documented data quality checks and cleaning decisions.

## Additional Comments

- The original raw dataset will remain unchanged.
- The geographical coverage and category definitions will be
  checked before cleaning.
- Missing values and duplicates will be investigated before
  deciding how to handle them.
- Economic inactivity and unemployment are different measures
  and will be kept distinct during preparation and analysis.

  ---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir("..")

print(os.getcwd())

c:\Dudley employment barriers


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Import Libraries

- os: manage the working directory and output folders.
- pandas: load, inspect, clean and save the dataset.

In [4]:
import os
import pandas as pd

---

## 2. Inspect the Raw Dataset

Check the first ten lines to identify column headings and any
introductory text before loading the CSV into pandas.

In [5]:
raw_path = "raw_datasets/dudley_economic_activity.csv"

with open(raw_path, encoding="utf-8-sig") as file:
    for _ in range(10):
        line = file.readline()
        if not line:
            break
        print(line.rstrip())


"TS066 - Economic activity status"
"ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
"Population :","All usual residents aged 16 years and over"
"Units      :","Persons"
"Date       :","2021"

"2021 super output area - lower layer","Total: All usual residents aged 16 years and over","Economically active (excluding full-time students)","Economically active (excluding full-time students):In employment","Economically active (excluding full-time students): Unemployed","Economically inactive: Student","Economically inactive: Looking after home or family","Economically inactive: Long-term sick or disabled","Economically inactive: Other"

"E01009763 : Dudley 001A",1637,970,888,82,97,164,73,70


### Findings from the Raw File Inspection

- The dataset is TS066: Economic activity status, from Census 2021.
- It covers all usual residents aged 16 years and over.
- Values represent counts of persons.
- The geography is 2021 Lower Layer Super Output Areas (LSOAs).
- The first six lines contain metadata and a blank line.
  These will be skipped when loading the dataset.
- The column headings appear on the seventh line.
- The first visible data record is Dudley 001A.

### 2.1 Load the Dataset

Load the CSV into a pandas DataFrame, skipping the six introductory
lines so the column headings are read correctly.

Display the dataset dimensions and the first and last five rows
to check the structure and identify any footer notes.

In [6]:
# Load the CSV, skipping the six introductory lines.
df = pd.read_csv(
    raw_path,
    skiprows=6,
    encoding="utf-8-sig"
)

# Show the dataset dimensions.
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Display the first five rows.
display(df.head())

Rows: 208, Columns: 9


,2021 super output area - lower layer,Total: All usual residents aged 16 years and over,Economically active (excluding full-time students),Economically active (excluding full-time students):In employment,Economically active (excluding full-time students): Unemployed,Economically inactive: Student,Economically inactive: Looking after home or family,Economically inactive: Long-term sick or disabled,Economically inactive: Other
0,E01009763 : Dudley 001A,1637,970.0,888.0,82.0,97.0,164.0,73.0,70.0
1,E01009764 : Dudley 001B,1264,759.0,731.0,28.0,35.0,50.0,35.0,31.0
2,E01009767 : Dudley 001C,1265,748.0,699.0,49.0,35.0,69.0,66.0,28.0
3,E01009769 : Dudley 001D,1431,701.0,618.0,83.0,78.0,119.0,148.0,59.0
4,E01009770 : Dudley 001E,1214,664.0,625.0,39.0,41.0,73.0,69.0,50.0


Section 2 content

### Findings from Loading and Inspecting the Full Dataset

- The imported dataset contains 208 rows and 9 columns.
- Rows 0–202 contain 203 Dudley LSOA records.
- Rows 203–206 contain disclosure-control notes.
- Row 207 is empty.
- Area codes and area names are combined in the first column.
- No missing values are visible within the 203 area records;
  this will be confirmed with a missing-value check.
- Count values are displayed with decimal places in several
  columns. Data types will be checked after removing non-data rows.
- The selected columns include an economically active subtotal
  and its employment and unemployment components. These must
  not be added together, as this would double-count people.
- The export does not include every economic activity category,
  so the selected categories should not be expected to sum
  to the total population.
- The source notes describe confidentiality adjustments to counts,
  which should be considered when interpreting small values.

The next step is to remove the five non-data rows while preserving
all 203 area records and keeping the original CSV unchanged.

---

## 3. Remove Non-Data Rows

Remove the four footer-note rows and the empty row by retaining
records with a valid Dudley LSOA code and name.

Create a separate working DataFrame, keeping the imported data
in df unchanged. Display all excluded rows and the row counts
to verify what was removed.

In [7]:
# Developed with assistance from ChatGPT; output reviewed before proceeding.
# Identify records containing a Dudley LSOA code and name.
area_column = df.columns[0]
area_mask = df[area_column].str.fullmatch(
    r"E\d{8}\s*:\s*Dudley \d{3}[A-Z]",
    na=False
)

# Show every excluded row so the removal can be checked.
removed_rows = df.loc[~area_mask]
print("EXCLUDED ROWS:")
print(removed_rows.to_string(index=True))

# Create a working copy containing only Dudley area records.
df_clean = df.loc[area_mask].copy().reset_index(drop=True)

# Report the number of records before and after filtering.
print(f"\nOriginal rows: {len(df)}")
print(f"Removed rows: {len(removed_rows)}")
print(f"Remaining area records: {len(df_clean)}")
print(f"Columns: {df_clean.shape[1]}")

EXCLUDED ROWS:
    2021 super output area - lower layer                          Total: All usual residents aged 16 years and over  Economically active (excluding full-time students)  Economically active (excluding full-time students):In employment  Economically active (excluding full-time students): Unemployed  Economically inactive: Student  Economically inactive: Looking after home or family  Economically inactive: Long-term sick or disabled  Economically inactive: Other
203                                  NaN    In order to protect against disclosure of personal information, records                                                 NaN                                                               NaN                                                             NaN                             NaN                                                  NaN                                                NaN                           NaN
204                                  NaN  have been swapp

### Findings from Removing Non-Data Rows

- Removed four disclosure-control note rows and one empty row.
- Retained all 203 Dudley LSOA records and all 9 columns.
- The excluded rows contain no area records.
- The working DataFrame, df_clean, now contains only area data,
  with its index reset.
- The imported DataFrame, df, and the original CSV remain unchanged.
- The disclosure-control explanation remains documented above
  and preserved in the raw file.

---

## 4. Check Missing Values, Duplicates and Data Types

Check all 203 area records for missing values, fully duplicated
rows and repeated area identifiers.

Inspect the data type of every column before converting counts
to numeric types. Report any affected rows for investigation
before making further changes.

In [8]:
# Developed with assistance from ChatGPT.

# Summarise missing values and data types for every column.
quality_summary = pd.DataFrame({
    "Data type": df_clean.dtypes.astype(str),
    "Missing values": df_clean.isna().sum()
})

print("COLUMN CHECKS:")
print(quality_summary.to_string())

# Identify all rows involved in duplicates.
duplicate_rows = df_clean.duplicated(keep=False)
duplicate_areas = df_clean.duplicated(
    subset=[area_column],
    keep=False
)

print(f"\nRows in duplicate groups: {duplicate_rows.sum()}")
print(f"Rows with repeated area identifiers: {duplicate_areas.sum()}")

# Display every record affected by missing values or duplicates.
affected_rows = (
    df_clean.isna().any(axis=1)
    | duplicate_rows
    | duplicate_areas
)

print("\nRECORDS REQUIRING INVESTIGATION:")
if affected_rows.any():
    print(df_clean.loc[affected_rows].to_string(index=True))
else:
    print("No missing values or duplicates found.")

COLUMN CHECKS:
                                                                 Data type  Missing values
2021 super output area - lower layer                                object               0
Total: All usual residents aged 16 years and over                   object               0
Economically active (excluding full-time students)                 float64               0
Economically active (excluding full-time students):In employment   float64               0
Economically active (excluding full-time students): Unemployed     float64               0
Economically inactive: Student                                     float64               0
Economically inactive: Looking after home or family                float64               0
Economically inactive: Long-term sick or disabled                  float64               0
Economically inactive: Other                                       float64               0

Rows in duplicate groups: 0
Rows with repeated area identifiers: 0

RECORD

### Findings from Data Quality Checks

- All 203 area records have no missing values across the 9 columns.
- No fully duplicated rows or repeated area identifiers were found.
- No records need to be removed or filled following these checks.
- The area identifier column has an object data type, suitable
  for its current text values.
- The total population column also has an object data type,
  because it originally included footer text.
- The remaining seven count columns have float64 data types.
- All eight count columns will be checked for numeric, whole-number
  and non-negative values before conversion to integer types.

---

## 5. Validate and Convert Count Columns

Check all eight count columns for non-numeric, missing, negative
or fractional values.

If every value passes these checks, convert the count columns
to integers because they represent numbers of people.
If any values fail, display the affected records for investigation
and leave the original columns unchanged.

In [9]:
# Developed with assistance from ChatGPT.

# Select the eight count columns, excluding the area identifier.
count_columns = df_clean.columns.drop(area_column)

# Create a numeric copy for validation.
numeric_counts = df_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Check every count for missing, non-numeric or invalid values.
invalid_counts = (
    numeric_counts.isna()
    | numeric_counts.lt(0)
    | numeric_counts.mod(1).ne(0)
)

print("INVALID VALUES PER COLUMN:")
print(invalid_counts.sum().to_string())

# Developed with assistance from ChatGPT.
# Convert only if all count values pass validation.
if invalid_counts.any().any():
    print("\nRECORDS REQUIRING INVESTIGATION:")
    print(
        df_clean.loc[invalid_counts.any(axis=1)].to_string(
            index=True
        )
    )
    print("\nNo column types were changed.")
else:
    df_clean = df_clean.astype({
        column: "int64" for column in count_columns
    })

    print("\nAll eight count columns converted to integers.")
    print("\nUPDATED DATA TYPES:")
    print(df_clean.dtypes.to_string())
    print(f"\nRows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

INVALID VALUES PER COLUMN:
Total: All usual residents aged 16 years and over                   0
Economically active (excluding full-time students)                  0
Economically active (excluding full-time students):In employment    0
Economically active (excluding full-time students): Unemployed      0
Economically inactive: Student                                      0
Economically inactive: Looking after home or family                 0
Economically inactive: Long-term sick or disabled                   0
Economically inactive: Other                                        0

All eight count columns converted to integers.

UPDATED DATA TYPES:
2021 super output area - lower layer                                object
Total: All usual residents aged 16 years and over                    int64
Economically active (excluding full-time students)                   int64
Economically active (excluding full-time students):In employment     int64
Economically active (excluding full-time stu

### Findings from Count Validation and Conversion

- All eight count columns passed validation across all 203 records.
- No missing, non-numeric, negative or fractional values were found.
- All eight count columns were successfully converted to int64.
- This removes unnecessary decimal places while preserving
  the original count values.
- The area identifier column remains text.
- The dataset still contains 203 rows and 9 columns.

---

## 6. Separate Area Codes and Names

Split the combined area identifier into LSOA_Code and LSOA_Name.
The code will be used to join this dataset with other datasets,
once their geographical boundaries have been checked.

Keep the original identifier for now and display all 203 mappings
to verify the split.

In [12]:
# Developed with assistance from ChatGPT.

# Split on the first colon and remove surrounding whitespace.
area_parts = df_clean[area_column].str.split(
    ":", n=1, expand=True
)

df_clean["LSOA_Code"] = area_parts[0].str.strip()
df_clean["LSOA_Name"] = area_parts[1].str.strip()

# Display the first five identifier mappings.
print("AREA CODE AND NAME MAPPINGS:")
display(
    df_clean[
        [area_column, "LSOA_Code", "LSOA_Name"]
    ].head()
)

# Check the separated identifiers across all records.
print("\nIDENTIFIER CHECKS:")
print(f"Unique LSOA codes: {df_clean['LSOA_Code'].nunique()}")
print(f"Missing codes: {df_clean['LSOA_Code'].isna().sum()}")
print(f"Missing names: {df_clean['LSOA_Name'].isna().sum()}")
print(f"Duplicate codes: {df_clean['LSOA_Code'].duplicated().sum()}")
print(f"Rows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

AREA CODE AND NAME MAPPINGS:


,2021 super output area - lower layer,LSOA_Code,LSOA_Name
0,E01009763 : Dudley 001A,E01009763,Dudley 001A
1,E01009764 : Dudley 001B,E01009764,Dudley 001B
2,E01009767 : Dudley 001C,E01009767,Dudley 001C
3,E01009769 : Dudley 001D,E01009769,Dudley 001D
4,E01009770 : Dudley 001E,E01009770,Dudley 001E



IDENTIFIER CHECKS:
Unique LSOA codes: 203
Missing codes: 0
Missing names: 0
Duplicate codes: 0
Rows: 203, Columns: 11


### Findings from Separating Area Codes and Names

- Successfully separated the area identifiers into LSOA_Code
  and LSOA_Name.
- The separated values match the original identifiers.
- All 203 LSOA codes are unique.
- No missing codes, missing names or duplicate codes were found.
- The original combined identifier has been retained for now.
- The dataset contains 203 rows and 11 columns after adding
  the two new fields.
- LSOA_Code provides a joining key for other datasets, subject
  to checking that their geographical boundaries are compatible.

---

## 7. Rename the columns

Shorten the column headings to make them easier to use in later
notebooks. Keep the age group and the exclusion of full-time students
clear in the relevant names.

Keep `LSOA_Code` and `LSOA_Name` consistent with the qualifications
dataset. Retain the original area identifier as `Original_Area`.

In [14]:
# Developed with assistance from ChatGPT.

# Match the original headings to shorter, descriptive names.
column_names = {
    "2021 super output area - lower layer": "Original_Area",
    "Total: All usual residents aged 16 years and over": (
        "Total_Aged_16_Plus"
    ),
    "Economically active (excluding full-time students)": (
        "Economically_Active_Excl_FT_Students"
    ),
    "Economically active (excluding full-time students):In employment": (
        "Employed_Excl_FT_Students"
    ),
    "Economically active (excluding full-time students): Unemployed": (
        "Unemployed_Excl_FT_Students"
    ),
    "Economically inactive: Student": "Inactive_Student",
    "Economically inactive: Looking after home or family": (
        "Inactive_Home_Or_Family"
    ),
    "Economically inactive: Long-term sick or disabled": (
        "Inactive_Long_Term_Sick_Or_Disabled"
    ),
    "Economically inactive: Other": "Inactive_Other"
}

# Apply the new headings without changing the data values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
df_clean = df_clean.rename(columns=column_names)

# Display the headings to check the changes.
print("COLUMN NAMES:")
for column in df_clean.columns:
    print(column)

print(f"\nRows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")

COLUMN NAMES:
Original_Area
Total_Aged_16_Plus
Economically_Active_Excl_FT_Students
Employed_Excl_FT_Students
Unemployed_Excl_FT_Students
Inactive_Student
Inactive_Home_Or_Family
Inactive_Long_Term_Sick_Or_Disabled
Inactive_Other
LSOA_Code
LSOA_Name

Rows: 203, Columns: 11


### Findings

The column headings now use shorter, descriptive names.
The relevant headings retain the exclusion of full-time students,
and `Total_Aged_16_Plus` identifies the population covered.

`LSOA_Code` and `LSOA_Name` match the names used in the qualifications
dataset. The original area identifier is retained as `Original_Area`.

The dataset still contains **203 rows and 11 columns**.
Only the headings changed; the data values remain unchanged.

---

## 8. Save the Cleaned Dataset

Save the current cleaned data as a separate CSV without the
DataFrame index. Keep the original column names and the separate
LSOA code and name columns.

Reload the saved file and compare it with df_clean to confirm
that the export preserves all values, columns and records.
The original raw CSV will remain unchanged.

In [15]:
# Developed with assistance from ChatGPT.

# Create the output folder if it does not already exist.
os.makedirs("cleaned_datasets", exist_ok=True)

# Save the cleaned data without adding an index column.
output_path = (
    "cleaned_datasets/dudley_economic_activity_cleaned.csv"
)
df_clean.to_csv(output_path, index=False)

# Reload the saved CSV and verify the complete dataset.
saved_df = pd.read_csv(output_path)
pd.testing.assert_frame_equal(saved_df, df_clean)

# Report the export and final quality checks.
print(f"Saved to: {output_path}")
print(f"Rows: {len(saved_df)}, Columns: {saved_df.shape[1]}")
print(f"Missing values: {saved_df.isna().sum().sum()}")
print(f"Duplicate rows: {saved_df.duplicated().sum()}")
print(f"Duplicate LSOA codes: {saved_df['LSOA_Code'].duplicated().sum()}")
print("Verification passed: saved data matches df_clean.")

Saved to: cleaned_datasets/dudley_economic_activity_cleaned.csv
Rows: 203, Columns: 11
Missing values: 0
Duplicate rows: 0
Duplicate LSOA codes: 0
Verification passed: saved data matches df_clean.


### Findings from Saving the Cleaned Dataset

- Saved the cleaned CSV to:
  `cleaned_datasets/dudley_economic_activity_cleaned.csv`
- The saved dataset contains 203 rows and 11 columns.
- No missing values, duplicate rows or duplicate LSOA codes remain.
- Reloading and comparing the saved file confirmed that it
  matches df_clean exactly.
- Original column names and the combined area identifier are
  retained alongside the separate LSOA_Code and LSOA_Name columns.
- The raw CSV remains unchanged.

Initial cleaning of the economic activity dataset is complete.
Further preparation will follow after cleaning the other datasets.

---

## Ethical Considerations — Cleaning Economic Activity Data

### Purpose and Responsible Use

This notebook prepares the Census 2021 economic activity data for later analysis of employment barriers in Dudley. Cleaning decisions should preserve the meaning of the original categories and make the data usable without introducing assumptions about residents’ circumstances or willingness to work.

Economic activity status describes a person’s recorded circumstances at the time of the Census. It should not be treated as a measure of personal worth, effort or potential.

### Reading the Source and Its Metadata

The source file contains introductory metadata before the table. Reading the correct header allows the code to distinguish actual observations from the dataset title, population description, units and source information.

Although these introductory lines are not retained as data rows, their meaning should remain documented. The population covers usual residents aged 16 years and over, the units are persons, and the observations refer to 2021. Losing this context could lead to incorrect comparisons or claims about working-age residents or current conditions.

ONS and Nomis attribution and the applicable licensing information should remain available in the project documentation.

### Removing Empty Rows and Footer Notes

Empty rows and explanatory footer text need to be separated from the neighbourhood records before calculations can be performed. These are formatting elements rather than observations about Dudley residents.

Removing non-data rows should not become a reason to discard valid areas with inconvenient or unusual values. Every removal should have a clear structural justification, and relevant source notes should remain available in the original file.

### Extracting LSOA Codes and Names

The raw file combines the LSOA code and neighbourhood name in one field. Separating these provides a consistent identifier for later matching while retaining a readable area name.

Accurate code extraction matters because an incorrect identifier could attach one neighbourhood’s economic activity figures to another area. Area codes should be checked for missing values and duplication before they are used in joins.

Matching codes alone does not establish that boundaries are compatible across different geographic releases. That question must be addressed separately during dataset integration.

### Renaming Columns

Shorter column names make the code easier to read, but they must preserve the original definitions. In particular, the economically active, employed and unemployed categories in this extract exclude full-time students.

Removing this distinction from names or descriptions could make a restricted category appear to represent every employed or unemployed resident. Renaming should improve clarity without broadening the meaning of a measure.

### Converting Counts to Numeric Values

Population counts need numeric data types for reliable calculations. Conversion should preserve the published values and reveal unexpected text or missing entries rather than silently replacing them with zero.

A missing value is not evidence that no residents belong to a category. Treating missing or unreadable values as zero could understate an area’s population or apparent support needs.

Counts should remain whole-person values. Unusual counts should be investigated against the source rather than altered simply because they differ from other neighbourhoods.

### Checking Missing Values and Duplicate Records

Missing-value and duplicate checks help prevent incomplete or repeated records from affecting later analysis. Duplicate LSOA codes could cause repeated rows during merging and give an area more influence than intended.

Any duplicate should be investigated before removal. Identical rows, repeated identifiers and conflicting observations are different problems and should not automatically receive the same treatment.

A dataset with no missing values or duplicates is structurally easier to analyse, but these checks do not establish that the original Census measurements are free from uncertainty.

### Respecting Category Relationships

The economic activity columns are not all separate groups that can be added together. Employment and unemployment are components of the economically active category excluding full-time students. Adding that total to its components would double-count residents.

This extract also contains selected inactivity categories rather than a complete breakdown of every resident’s economic activity status. Retirement and some student-related categories are not included.

Any arithmetic check should therefore reflect the actual relationship between the selected categories. Their combined sum should not be assumed to equal the total population aged 16 and over.

### Avoiding Misleading Rates

The cleaned file preserves counts for later preparation. Any subsequent percentage must use a denominator appropriate to the question and retain a clear label.

Unemployed residents as a percentage of all residents aged 16 and over is not the same measure as an unemployment rate based on the economically active population. Similarly, selected inactivity categories do not provide a complete economic inactivity rate.

The population aged 16 and over includes people beyond conventional working ages. It should not be described as a working-age population.

### Interpreting Inactivity Respectfully

Looking after home or family, long-term sickness or disability, studying and other forms of inactivity represent different circumstances. They should not be grouped or described in ways that imply a shared cause or a lack of willingness to work.

Caring responsibilities may involve substantial unpaid work. Health limitations may interact with workplace accessibility, support and available opportunities. The recorded category alone cannot explain an individual’s experience.

The label “other” should retain its uncertainty rather than being assigned an explanation that the source does not provide.

### Geographic and Time Limitations

The cleaned data contains 203 Dudley Census 2021 LSOAs. These neighbourhood records should be preserved at this stage, with any later exclusions for geographic compatibility explained in the merging notebook.

Census 2021 figures describe a particular historical period. They should not be presented as current employment conditions without additional evidence, and the wider circumstances of that period should be considered when interpreting patterns.

### Area-Level Interpretation and Privacy

The records describe neighbourhood populations, not identifiable individuals. Later analysis should remain at an appropriate aggregate level and should not attempt to infer the economic activity status of particular residents.

An area with a higher count or percentage in a category does not mean that every resident shares that characteristic. Labels, charts and conclusions should avoid stigmatising neighbourhoods or presenting area averages as individual facts.

### Saving the Cleaned Dataset

Saving the cleaned data separately from the raw source supports traceability. The original file should remain unchanged so that decisions can be reviewed and values can be checked against their published source.

The cleaned output should retain the identifiers, counts and category distinctions needed for later validation. Saving successfully confirms that a file was produced; it does not, by itself, confirm that every cleaning decision was correct.

### Transparency and Accountability

Code comments, findings and project documentation should explain the cleaning decisions and their limitations. References should support the methods actually used, and ChatGPT assistance should be acknowledged transparently.

Responsibility for interpretation remains with the analyst. Later conclusions should be based on the documented definitions and observed evidence, without presenting associations as proof of the causes of unemployment or inactivity.